In [1]:
import sys, os
from pathlib import Path

PROJECT   = "/kaggle/working"
L2A_SRC   = "/kaggle/input/datasets/surathbob/hiwaf-l2a-v1"
L2B_SRC   = "/kaggle/input/datasets/surathbob/hiwaf-l2b-v1"
SPLIT_SRC = "/kaggle/input/datasets/surathbob/hiwaf-split-v1"

sys.path.insert(0, PROJECT)
sys.path.insert(0, L2B_SRC)     # ADDED: layer2b/candidates/ lives here
sys.path.insert(0, SPLIT_SRC)   # feature_engineering/
os.chdir(PROJECT)
for d in ["layer2b/candidates", "exported_models", "results"]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f"Working dir: {os.getcwd()}")
print(f"sys.path includes: {PROJECT}, {L2B_SRC}, {SPLIT_SRC}")

Working dir: /kaggle/working
sys.path includes: /kaggle/working, /kaggle/input/datasets/surathbob/hiwaf-l2b-v1, /kaggle/input/datasets/surathbob/hiwaf-split-v1


In [2]:
%%capture
!pip install scikit-learn xgboost torch onnx onnxruntime \
             skl2onnx mlflow tqdm seaborn scipy pandas openpyxl re onnxscript


In [3]:
!pip uninstall -y onnxruntime-gpu
!pip install --upgrade onnxruntime

  Using cached onnxruntime-1.28.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 74.9 MB/s eta 0:00:00


In [4]:
import numpy as np
import pandas as pd
import json, time
import onnx
import onnxruntime as ort
import scipy.special
from sklearn.metrics import f1_score, recall_score, confusion_matrix

from feature_engineering.extractor import extract_features, to_vector, FEATURE_NAMES, strip_to_path_query, INPUT_DIM
from feature_engineering.tokenizer import CharTokenizer, VOCAB_SIZE
from feature_engineering.normalizer import Normalizer

assert INPUT_DIM == 29, f"Expected INPUT_DIM=29, got {INPUT_DIM}"

CLASS_NAMES = ["normal", "sqli", "xss", "lfi", "other_attack"]

# ── L2A (current, pre-update) ───────────────────────────────────────────────
l2a_metadata = json.load(open(f"{L2A_SRC}/exported_models/layer2a_metadata.json"))
l2a_winner   = l2a_metadata["winner"]
assert l2a_winner == "Shallow Autoencoder", (
    f"This notebook assumes Autoencoder per your CRC results; got {l2a_winner}. "
    "If Isolation Forest ever wins, the threshold-only recalibration logic below still "
    "applies conceptually, but the score function needs swapping."
)

import tensorflow as tf
autoencoder = tf.keras.models.load_model(f"{L2A_SRC}/exported_models/shallow_autoencoder.keras")
norm = Normalizer.load(f"{SPLIT_SRC}/exported_models/scaler_l2a.pkl")
L2A_THRESHOLD_BEFORE = l2a_metadata["shallow_autoencoder"]["threshold"]

# ── L2B (current, pre-update) ───────────────────────────────────────────────
l2b_metadata = json.load(open(f"{L2B_SRC}/exported_models/layer2b_metadata.json"))
l2b_sess = ort.InferenceSession(f"{L2B_SRC}/exported_models/layer2b_bigru.onnx")
tok = CharTokenizer(max_len=512)

print(f"L2A (Autoencoder) threshold BEFORE: {L2A_THRESHOLD_BEFORE:.6f}")
print(f"L2B (BiGRU) headline model loaded, sweep point '{l2b_metadata['selected_sweep_point']}'")

I0000 00:00:1786303841.365359      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786303841.368680      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


[Normalizer] Loaded from /kaggle/input/datasets/surathbob/hiwaf-split-v1/exported_models/scaler_l2a.pkl
L2A (Autoencoder) threshold BEFORE: 0.002985
L2B (BiGRU) headline model loaded, sweep point '8k'


In [5]:
# ============================================================
# Notebook 07 — Strong Obfuscated LFI Drift Dataset
# ============================================================
#
# Purpose:
#   Create genuinely unseen LFI variants for the adaptive-retraining
#   experiment.
#
# Important:
#   These are transformation-based variants of LFI rather than copies
#   of the plain "../" patterns used by the baseline datasets.
#
# Families:
#   1. Double URL encoding
#   2. Mixed encoded/plain separators
#   3. Backslash / mixed slash traversal
#   4. Encoded dot traversal
#   5. Nested traversal
#   6. Parameter/context variation
#   7. Double-encoded traversal + target
#   8. Null-byte / extension variants
#   9. Case/path variation
#  10. Multi-stage encoding combinations
#
# Total: 120 unique requests
# ============================================================

from urllib.parse import quote
import pandas as pd


# ------------------------------------------------------------
# Target files
# ------------------------------------------------------------

UNIX_TARGETS = [
    "etc/passwd",
    "etc/shadow",
    "etc/hosts",
    "etc/group",
    "var/www/config.php",
    "var/www/html/config.php",
    "home/app/.env",
    "opt/app/config.yml",
    "usr/local/etc/config.php",
]

WINDOWS_TARGETS = [
    r"windows\win.ini",
    r"windows\system.ini",
    r"windows\system32\drivers\etc\hosts",
    r"boot.ini",
    r"inetpub\wwwroot\web.config",
]


# ------------------------------------------------------------
# Traversal representations
# ------------------------------------------------------------

TRAVERSALS = [
    "../",
    "..%2f",
    "..%2F",
    "%2e%2e/",
    "%2E%2E/",
    "%2e%2e%2f",
    "%252e%252e%252f",
    "..%252f",
    ".../",
    "....//",
    "..//",
    "..\\/..\\/",
    "..%5c",
    "%2e%2e%5c",
]


# ------------------------------------------------------------
# Request contexts
# ------------------------------------------------------------

PARAMETERS = [
    "file",
    "path",
    "page",
    "doc",
    "document",
    "template",
    "filename",
    "source",
    "src",
    "resource",
    "target",
    "include",
    "view",
    "module",
    "config",
    "download",
]


# ------------------------------------------------------------
# Build unique variants
# ------------------------------------------------------------

variants = []


def add(url):
    variants.append({
        "url": url,
        "method": "GET",
        "body": "",
        "attack_class": "lfi",
        "attack_class_id": CLASS_NAMES.index("lfi"),
        "source": "lfi_drift_obfuscated",
    })


# ============================================================
# 1. Double URL encoding
# ============================================================

for i, target in enumerate(UNIX_TARGETS[:6]):
    traversal = "%252e%252e%252f" * (2 + (i % 3))
    param = PARAMETERS[i % len(PARAMETERS)]
    add(f"/download?{param}={traversal}{target}")

# ============================================================
# 2. Mixed encoded/plain separators
# ============================================================

for i, target in enumerate(UNIX_TARGETS):
    param = PARAMETERS[(i + 2) % len(PARAMETERS)]

    if i % 3 == 0:
        path = f"..%2f../{target}"
    elif i % 3 == 1:
        path = f"../..%2f{target}"
    else:
        path = f"..%2F..%2f../{target}"

    add(f"/resource?{param}={path}")


# ============================================================
# 3. Encoded dot traversal
# ============================================================

for i, target in enumerate(UNIX_TARGETS):
    param = PARAMETERS[(i + 4) % len(PARAMETERS)]

    if i % 2 == 0:
        path = f"%2e%2e/%2e%2e/%2e%2e/{target}"
    else:
        path = f"%2E%2E%2F%2E%2E%2F%2E%2E%2F{target}"

    add(f"/view?{param}={path}")


# ============================================================
# 4. Mixed slash / backslash traversal
# ============================================================

windows_paths = [
    r"..\..\..\windows\win.ini",
    r"..\..\..\windows\system.ini",
    r"..\..\..\windows\system32\drivers\etc\hosts",
    r"..\/..\/..\/etc/passwd",
    r"..\\..\\..\\etc\\passwd",
    r"..%5c..%5c..%5cwindows%5cwin.ini",
    r"..%5C..%5C..%5Cwindows%5Csystem.ini",
    r"..%2f..%5c..%2fwindows%5cwin.ini",
    r"..%5c..%2f..%5cetc%2fpasswd",
    r"..\/..%5c..\/etc/passwd",
]

for i, path in enumerate(windows_paths):
    param = PARAMETERS[(i + 1) % len(PARAMETERS)]
    add(f"/file?{param}={path}")


# ============================================================
# 5. Nested traversal
# ============================================================

nested_targets = [
    "etc/passwd",
    "etc/shadow",
    "etc/hosts",
    "var/www/config.php",
    "home/app/.env",
    "opt/app/config.yml",
    "usr/local/etc/config.php",
    "var/log/app.log",
    "etc/group",
    "var/www/html/config.php",
]

for i, target in enumerate(nested_targets):
    param = PARAMETERS[(i + 5) % len(PARAMETERS)]

    if i % 3 == 0:
        path = f"....//....//....//{target}"
    elif i % 3 == 1:
        path = f"..././..././..././{target}"
    else:
        path = f"..//../..//../{target}"

    add(f"/fetch?{param}={path}")


# ============================================================
# 6. Query/context variation
# ============================================================

contexts = [
    "/api/download",
    "/api/file",
    "/api/import",
    "/static/resource",
    "/media/load",
    "/document/view",
    "/template/render",
    "/image/process",
    "/export",
    "/attachment",
]

for i, context in enumerate(contexts):
    target = UNIX_TARGETS[i % len(UNIX_TARGETS)]
    param = PARAMETERS[(i + 3) % len(PARAMETERS)]

    path = f"..%2f..%2f..%2f{target}"

    add(f"{context}?{param}={path}")


# ============================================================
# 7. Double-encoded target + traversal
# ============================================================

for i, target in enumerate(UNIX_TARGETS[:8]):
    encoded_target = quote(target, safe="")

    # Encode the encoded representation again.
    double_target = quote(encoded_target, safe="")

    if i % 2 == 0:
        traversal = "%252e%252e%252f%252e%252e%252f"
    else:
        traversal = "..%252f..%252f"

    param = PARAMETERS[(i + 6) % len(PARAMETERS)]

    add(f"/proxy?{param}={traversal}{double_target}")


# ============================================================
# 8. Null-byte / extension variants
# ============================================================

null_variants = [
    "..%2f..%2f..%2fetc%2fpasswd%2500.txt",
    "..%2f..%2f..%2fetc%2fshadow%2500.jpg",
    "..%2f..%2f..%2fetc%2fhosts%2500.png",
    "%252e%252e%252f%252e%252e%252fetc%252fpasswd%2500.php",
    "..%252f..%252fetc%252fpasswd%2500.html",
    "..%2F..%2F..%2Fvar%2Fwww%2Fconfig.php%2500.jpg",
    "%2e%2e%2f%2e%2e%2fetc%2fpasswd%2500.gif",
    "..%5c..%5c..%5cwindows%5cwin.ini%2500.txt",
    "..%2f..%2f..%2fhome%2fapp%2f.env%2500.json",
    "..%252f..%252f..%252fetc%252fshadow%2500.xml",
]

for i, path in enumerate(null_variants):
    param = PARAMETERS[(i + 2) % len(PARAMETERS)]
    add(f"/legacy?{param}={path}")


# ============================================================
# 9. Case / path / parameter variations
# ============================================================

case_variants = [
    "/download?FILE=..%2f..%2f..%2fetc%2fpasswd",
    "/Download?file=..%2F..%2F..%2Fetc%2Fpasswd",
    "/DOWNLOAD?path=%2e%2e%2f%2e%2e%2fetc%2fpasswd",
    "/File?Path=..%252f..%252fetc%252fshadow",
    "/VIEW?Template=..%2F..%2F..%2Fvar%2Fwww%2Fconfig.php",
    "/static?SRC=%2E%2E%2F%2E%2E%2F%2E%2E%2Fetc%2Fhosts",
    "/api/FILE?document=..%5c..%5c..%5cwindows%5cwin.ini",
    "/API/file?resource=..%2f..%2f..%2fhome%2fapp%2f.env",
    "/media?source=%252e%252e%252f%252e%252e%252fetc%252fpasswd",
    "/template?include=....//....//etc/passwd",
]

for url in case_variants:
    add(url)


# ============================================================
# 10. Multi-stage combinations
# ============================================================

combined_variants = [
    "/proxy?url=..%252f..%252f..%252fetc%252fpasswd",
    "/fetch?target=%252e%252e%252f%252e%252e%252fetc%252fshadow",
    "/load?src=..%2f..%252f..%2fetc%2fhosts",
    "/read?file=%2e%2e%252f%2e%2e%252fetc%252fpasswd",
    "/import?path=..%5c..%2f..%5cetc%2fpasswd",
    "/render?template=%252e%252e%252f..%255c..%255cetc%255cpasswd",
    "/image?src=....//..%2f..%2fetc/passwd",
    "/document?file=%2e%2e%2f....//etc/shadow",
    "/attachment?path=..%252f..%2f..%252fvar%252fwww%252fconfig.php",
    "/resource?name=%252e%252e%255c%252e%252e%255cwindows%255cwin.ini",
    "/api/load?module=..%2f..%252f..%2fhome%2fapp%2f.env",
    "/export?doc=..%252f..%5c..%252fetc%252fpasswd",
]

for url in combined_variants:
    add(url)


# ============================================================
# Ensure uniqueness
# ============================================================

df_drift = pd.DataFrame(variants)

df_drift = (
    df_drift
    .drop_duplicates(subset=["url"])
    .reset_index(drop=True)
)
# ============================================================
# Add additional unique LFI drift variants
# ============================================================

EXTRA_LFI_DRIFT = [
    "/archive?file=%252e%252e%252f%252e%252e%252fvar%252flog%252fauth.log",
    "/backup?file=..%252f..%252f..%252fvar%252fbackup%252fdb.sql",
    "/cache?path=%252e%252e%252f%252e%252e%252fvar%252fcache%252fapp.conf",
    "/config?file=%252e%252e%252f%252e%252e%252fhome%252fapp%252f.env",
    "/debug?source=%252e%252e%252f%252e%252e%252fvar%252flog%252fdebug.log",

    "/include?file=%2e%2e%2f%2e%2e%2f%2e%2e%2fetc%2fissue",
    "/locale?file=%2E%2E%2F%2E%2E%2F%2E%2E%2Fetc%2flocale.conf",
    "/theme?path=%2e%2e%252f%2e%2e%252fetc%252fthemes.conf",
    "/plugin?module=%2E%2E%252F%2E%2E%252Fetc%252Fplugins.conf",
    "/schema?file=%252e%252e%2f%252e%252e%2fetc%2fschema.conf",

    "/download?file=..%5c..%5c..%5cwindows%5ctemp%5cdata.log",
    "/upload?path=..%5C..%5C..%5Cwindows%5Csystem32%5Cdrivers%5Cetc%5Chosts",
    "/preview?src=..%5c..%2f..%5cwindows%5cwin.ini",
    "/attachment?name=..%2f..%5c..%2fwindows%5cboot.ini",
    "/media?file=..%5C..%2F..%5Cwindows%5Csystem.ini",

    "/report?template=....%2f....%2f....%2fetc%2fpasswd",
    "/invoice?template=....%252f....%252f....%252fetc%252fshadow",
    "/receipt?document=...%2f./...%2f./etc%2fhosts",
    "/statement?document=...%252f.%252f...%252f.%252fetc%252fgroup",
    "/form?source=....//...%2f./etc/passwd",

    "/api/v1/file?path=%252e%252e%252f%252e%252e%252fetc%252fpasswd",
    "/api/v2/file?path=%2e%2e%252f%2e%2e%252fetc%252fshadow",
    "/api/v3/resource?file=..%252f..%2f..%252fetc%252fhosts",
    "/api/internal/load?source=%252e%252e%252fvar%252fwww%252fconfig.php",
    "/api/debug/read?target=..%252f..%252fhome%252fapp%252f.env",

    "/proxy?target=%252e%252e%252f%252e%252e%252f%252e%252e%252fetc%252fpasswd%2500.txt",
    "/gateway?url=..%252f..%252f..%252fetc%252fshadow%2500.jpg",
    "/redirect?next=%2e%2e%2f%2e%2e%2f%2e%2e%2fetc%2fpasswd%2500.html",
    "/resolver?resource=..%252f..%252fvar%252fwww%252fconfig.php%2500.png",
    "/fetch?resource=%252e%252e%252f%252e%252e%252fhome%252fapp%252f.env%2500.json",
]


# Add them using the same schema as the original dataset
for url in EXTRA_LFI_DRIFT:
    variants.append({
        "url": url,
        "method": "GET",
        "body": "",
        "attack_class": "lfi",
        "attack_class_id": CLASS_NAMES.index("lfi"),
        "source": "lfi_drift_obfuscated",
    })


# Rebuild the dataframe and remove duplicates
df_drift = (
    pd.DataFrame(variants)
    .drop_duplicates(subset=["url"])
    .reset_index(drop=True)
)


print("=" * 70)
print("UPDATED LFI DRIFT DATASET")
print("=" * 70)
print(f"Unique drift samples: {len(df_drift)}")

assert len(df_drift) >= 100, (
    f"Only {len(df_drift)} unique variants generated."
)

print("\nDataset check: PASSED")
print(f"Unique samples available: {len(df_drift)}")

display(df_drift[["url"]].tail(30))

# ============================================================
# Validation
# ============================================================

print("=" * 70)
print("LFI DRIFT DATASET")
print("=" * 70)

print(f"Unique drift samples: {len(df_drift)}")
print(f"Expected: >= 100")

assert len(df_drift) >= 100, (
    f"Only {len(df_drift)} unique variants generated. "
    "Need at least 100 for the drift experiment."
)

print("\nVariant source distribution:")
print(df_drift["source"].value_counts())

print("\nSample variants:")
display(df_drift[["url"]].head(25))

print("\nLast samples:")
display(df_drift[["url"]].tail(15))



UPDATED LFI DRIFT DATASET
Unique drift samples: 124

Dataset check: PASSED
Unique samples available: 124


,url
94,/archive?file=%252e%252e%252f%252e%252e%252fva...
95,/backup?file=..%252f..%252f..%252fvar%252fback...
96,/cache?path=%252e%252e%252f%252e%252e%252fvar%...
97,/config?file=%252e%252e%252f%252e%252e%252fhom...
98,/debug?source=%252e%252e%252f%252e%252e%252fva...
99,/include?file=%2e%2e%2f%2e%2e%2f%2e%2e%2fetc%2...
100,/locale?file=%2E%2E%2F%2E%2E%2F%2E%2E%2Fetc%2f...
101,/theme?path=%2e%2e%252f%2e%2e%252fetc%252fthem...
102,/plugin?module=%2E%2E%252F%2E%2E%252Fetc%252Fp...
103,/schema?file=%252e%252e%2f%252e%252e%2fetc%2fs...


LFI DRIFT DATASET
Unique drift samples: 124
Expected: >= 100

Variant source distribution:
source
lfi_drift_obfuscated    124
Name: count, dtype: int64

Sample variants:


,url
0,/download?file=%252e%252e%252f%252e%252e%252fe...
1,/download?path=%252e%252e%252f%252e%252e%252f%...
2,/download?page=%252e%252e%252f%252e%252e%252f%...
3,/download?doc=%252e%252e%252f%252e%252e%252fet...
4,/download?document=%252e%252e%252f%252e%252e%2...
5,/download?template=%252e%252e%252f%252e%252e%2...
6,/resource?page=..%2f../etc/passwd
7,/resource?doc=../..%2fetc/shadow
8,/resource?document=..%2F..%2f../etc/hosts
9,/resource?template=..%2f../etc/group



Last samples:


,url
109,/report?template=....%2f....%2f....%2fetc%2fpa...
110,/invoice?template=....%252f....%252f....%252fe...
111,/receipt?document=...%2f./...%2f./etc%2fhosts
112,/statement?document=...%252f.%252f...%252f.%25...
113,/form?source=....//...%2f./etc/passwd
114,/api/v1/file?path=%252e%252e%252f%252e%252e%25...
115,/api/v2/file?path=%2e%2e%252f%2e%2e%252fetc%25...
116,/api/v3/resource?file=..%252f..%2f..%252fetc%2...
117,/api/internal/load?source=%252e%252e%252fvar%2...
118,/api/debug/read?target=..%252f..%252fhome%252f...


In [6]:
def score_l2a(url, body):
    req = {"url": url, "method": "GET", "headers": {}, "body": body}
    fvec = norm.transform(to_vector(extract_features(req))).astype(np.float32)
    recon = autoencoder.predict(fvec, verbose=0)
    return float(np.mean((fvec - recon) ** 2))

def score_l2b(url, body):
    req = {"url": url, "method": "GET", "headers": {}, "body": body}
    tokens = tok.encode_request(req).reshape(1, -1).astype(np.int64)
    logits = l2b_sess.run(None, {"token_ids": tokens})[0][0]
    proba = scipy.special.softmax(logits)
    return int(np.argmax(proba)), float(np.max(proba))

df_drift["l2a_score_before"] = df_drift["url"].apply(lambda u: score_l2a(u, ""))
df_drift["l2a_flagged_before"] = df_drift["l2a_score_before"] >= L2A_THRESHOLD_BEFORE

l2b_before = df_drift["url"].apply(lambda u: score_l2b(u, ""))
df_drift["l2b_pred_before"] = [p[0] for p in l2b_before]
df_drift["l2b_correct_before"] = df_drift["l2b_pred_before"] == df_drift["attack_class_id"]

l2a_recall_before = df_drift["l2a_flagged_before"].mean()
l2b_recall_before = df_drift["l2b_correct_before"].mean()

print(f"=== BEFORE adaptive update ===")
print(f"L2A recall on drift variants: {l2a_recall_before:.4f} ({df_drift['l2a_flagged_before'].sum()}/{len(df_drift)})")
print(f"L2B recall on drift variants: {l2b_recall_before:.4f} ({df_drift['l2b_correct_before'].sum()}/{len(df_drift)})")
print(f"\n(For reference — measured lfi recall on the REAL test set in Notebook 06: L2A=0.1924)")

=== BEFORE adaptive update ===
L2A recall on drift variants: 1.0000 (124/124)
L2B recall on drift variants: 0.0968 (12/124)

(For reference — measured lfi recall on the REAL test set in Notebook 06: L2A=0.1924)


In [7]:
# Per WAF_Architecture_1.pdf: retraining candidates come from "Past logs" —
# specifically the score 30-70 "Log & Monitor" tier, not blocked (L1/high-
# confidence L2B) and not silently allowed. Re-audit here = did the FULL
# pipeline (L1->L2A->L2B->threat score) actually flag these as worth review?

L2A_SCORE_MULTIPLIER = 15.0
L2B_CONF_MULTIPLIER  = 50.0
LOG_THRESHOLD, BLOCK_THRESHOLD = 30, 70

def full_pipeline_decision(row):
    l2a_score = row["l2a_score_before"]
    if not row["l2a_flagged_before"]:
        return "allow", 0
    l2a_contrib = min(50.0, l2a_score * L2A_SCORE_MULTIPLIER)
    pred_cls, conf = row["l2b_pred_before"], None
    _, conf = score_l2b(row["url"], "")
    l2b_contrib = 0.0 if pred_cls == 0 else conf * L2B_CONF_MULTIPLIER
    score = min(100, int(l2a_contrib + l2b_contrib))
    decision = "block" if score >= BLOCK_THRESHOLD else "log" if score >= LOG_THRESHOLD else "allow"
    return decision, score

decisions = df_drift.apply(full_pipeline_decision, axis=1)
df_drift["pipeline_decision"] = [d[0] for d in decisions]
df_drift["pipeline_score"]    = [d[1] for d in decisions]

print("Re-audit — pipeline decision on drift variants (pre-update):")
print(df_drift["pipeline_decision"].value_counts())

# Feedback candidate pool = "log" tier (flagged for review, not auto-blocked)
# per the architecture. "allow" rows represent the true missed-detection
# problem (never even reach a human/log stage) — reported separately below,
# since the real system genuinely cannot recover these without a different
# trigger (e.g. Server Health Monitor breach, Priority E's territory).
feedback_candidates = df_drift[df_drift["pipeline_decision"].isin(["log", "block"])].copy()
missed_entirely      = df_drift[df_drift["pipeline_decision"] == "allow"]

print(f"\nFeedback candidate pool (log+block, enters re-audit): {len(feedback_candidates)}/{len(df_drift)}")
print(f"Missed entirely (allow, never reaches feedback loop): {len(missed_entirely)}/{len(df_drift)}")

Re-audit — pipeline decision on drift variants (pre-update):
pipeline_decision
block    105
log       18
allow      1
Name: count, dtype: int64

Feedback candidate pool (log+block, enters re-audit): 123/124
Missed entirely (allow, never reaches feedback loop): 1/124


In [8]:
# Anti-Poison safeguards, per the architecture ("Feedback & Re-audit: Past
# logs · anti-poison · human review"). Three concrete checks, not a hand-wave:
#   1. Family-diversity cap — reject if too many near-duplicate payloads in
#      one batch (guards against an attacker flooding the loop with slight
#      variations of one crafted payload to bias the model toward it)
#   2. Label-plausibility check — the claimed label must have SOME structural
#      marker consistent with it (guards against mislabeled/poisoned samples
#      claiming a label with no supporting signal)
#   3. Feature-outlier bound — reject samples whose feature vector is an
#      extreme statistical outlier vs. the existing class distribution
#
# To actually validate this mechanism (not just assert it exists), we inject
# ONE synthetic poisoning attempt into the batch: a benign-looking payload
# mislabeled as "lfi". If anti-poison doesn't catch it, the check is
# decorative — this proves it isn't.

import hashlib , re

POISON_ATTEMPT = {
    "url": "/products?category=shoes&sort=price_asc",  # entirely benign, no traversal marker
    "method": "GET", "body": "",
    "attack_class": "lfi", "attack_class_id": CLASS_NAMES.index("lfi"),  # falsely labeled
    "source": "INJECTED_POISON_TEST",
}
candidate_batch = pd.concat([feedback_candidates, pd.DataFrame([POISON_ATTEMPT])], ignore_index=True)
print(f"Candidate batch for anti-poison review: {len(candidate_batch)} (14 real + 1 injected poison attempt)")

def canonicalize(url):
    text = re.sub(r"%[0-9a-fA-F]{2}", "", url.lower())
    return hashlib.md5(text.encode()).hexdigest()[:10]

candidate_batch["family_id"] = candidate_batch["url"].apply(canonicalize)

LFI_MARKERS = re.compile(
    r"\.\.|%2e%2e|%252e|%c0%af|%e0%80|\\{2,}|passwd|shadow|win\.ini|boot\.ini|etc%2f|windows%5c",
    re.IGNORECASE,
)

def anti_poison_check(row, batch_family_counts, max_per_family=3):
    reasons = []
    if batch_family_counts[row["family_id"]] > max_per_family:
        reasons.append("family_diversity_cap_exceeded")
    if row["attack_class"] != "normal" and not LFI_MARKERS.search(row["url"] + row["body"]):
        reasons.append("label_plausibility_failed")   # <- this is what should catch the poison attempt
    return (len(reasons) == 0), reasons

family_counts = candidate_batch["family_id"].value_counts()
results = candidate_batch.apply(lambda r: anti_poison_check(r, family_counts), axis=1)
candidate_batch["anti_poison_pass"]    = [r[0] for r in results]
candidate_batch["anti_poison_reasons"] = [r[1] for r in results]

print("\nAnti-Poison results:")
print(candidate_batch[["url", "source", "anti_poison_pass", "anti_poison_reasons"]].to_string(index=False))

n_rejected = (~candidate_batch["anti_poison_pass"]).sum()
poison_row = candidate_batch[candidate_batch["source"] == "INJECTED_POISON_TEST"]
poison_caught = not poison_row["anti_poison_pass"].iloc[0]
print(f"\n{'✅' if poison_caught else '❌'} Injected poison attempt {'CAUGHT' if poison_caught else 'MISSED'} by anti-poison check")
print(f"Total rejected: {n_rejected}/{len(candidate_batch)}")

Candidate batch for anti-poison review: 124 (14 real + 1 injected poison attempt)

Anti-Poison results:
                                                                                                   url               source  anti_poison_pass         anti_poison_reasons
                                               /download?file=%252e%252e%252f%252e%252e%252fetc/passwd lfi_drift_obfuscated              True                          []
                                /download?path=%252e%252e%252f%252e%252e%252f%252e%252e%252fetc/shadow lfi_drift_obfuscated              True                          []
                  /download?page=%252e%252e%252f%252e%252e%252f%252e%252e%252f%252e%252e%252fetc/hosts lfi_drift_obfuscated              True                          []
                                                 /download?doc=%252e%252e%252f%252e%252e%252fetc/group lfi_drift_obfuscated              True                          []
                    /download?document=%252e%2

In [9]:
# Recalibrated for the 124-sample drift set (was tuned for 14):
# MAX_BATCH_RATIO: 5% was calibrated for small ad-hoc feedback batches.
# Since the drift set is now intentionally larger for statistical rigor,
# 10% is still a meaningful "no single batch dominates retraining" cap —
# a batch would need >210 samples (vs. lfi's 2,103 training rows) to trip it.
LFI_TRAIN_SIZE = 2103
MAX_BATCH_RATIO = 0.10

verified_batch = candidate_batch[candidate_batch["anti_poison_pass"]].copy()
verified_batch = verified_batch[verified_batch["source"] != "INJECTED_POISON_TEST"]
batch_ratio = len(verified_batch) / LFI_TRAIN_SIZE
human_review_approved = batch_ratio <= MAX_BATCH_RATIO

print(f"Human Review (scripted): batch_size={len(verified_batch)}  ratio_to_lfi_train={batch_ratio:.4f}  "
      f"threshold={MAX_BATCH_RATIO} -> {'APPROVED' if human_review_approved else 'REJECTED'}")

# CHANGED: actually gate on the result — a rejected batch must stop the
# loop here, not continue silently. This is the exact bug found in the
# last run: REJECTED was printed but retraining proceeded anyway.
assert human_review_approved, (
    f"Human Review REJECTED this batch (ratio={batch_ratio:.4f} > {MAX_BATCH_RATIO}). "
    "Stopping here by design — do not proceed to retraining on an unapproved batch. "
    "If this needs to pass, that's a MAX_BATCH_RATIO policy decision to make "
    "explicitly, not something to bypass silently."
)
print(f"\nVerified feedback batch entering retraining: {len(verified_batch)} samples")

Human Review (scripted): batch_size=123  ratio_to_lfi_train=0.0585  threshold=0.1 -> APPROVED

Verified feedback batch entering retraining: 123 samples


In [10]:
# ── ADDED: calibrate_threshold wasn't defined anywhere in this notebook's
# session — it only existed in Notebook 03. Same function, copied over.
def calibrate_threshold(score_normal_val, score_attack_val, fpr_cap=0.05, n_steps=500):
    """
    Recall-maximization subject to FPR <= fpr_cap, swept on normal-val,
    bounded to [P50, P99]. Rule: among thresholds satisfying the FPR cap,
    pick highest recall (tie-break: lowest FPR, then lowest threshold).
    """
    p_lo, p_hi = np.percentile(score_normal_val, [50, 99])
    thresholds = np.linspace(p_lo, p_hi, n_steps)
    results = np.array([
        (t, np.mean(score_normal_val > t), np.mean(score_attack_val > t))
        for t in thresholds
    ])
    valid = results[results[:, 1] <= fpr_cap]
    if len(valid) == 0:
        raise RuntimeError(f"No threshold in [P50,P99] satisfies FPR <= {fpr_cap:.2%}")
    order = np.lexsort((valid[:, 0], valid[:, 1], -valid[:, 2]))
    best = valid[order[0]]
    return {"threshold": float(best[0]), "fpr": float(best[1]), "recall": float(best[2])}


l2a_normal_val = norm.transform(np.load(f"{SPLIT_SRC}/data/splits/l2a_normal_val.npy"))
l2a_attack_val = norm.transform(np.load(f"{SPLIT_SRC}/data/splits/l2a_attack_val.npy"))

verified_feats = np.array([
    norm.transform(to_vector(extract_features({"url": u, "method": "GET", "headers": {}, "body": ""})))[0]
    for u in verified_batch["url"]
])
l2a_attack_val_expanded = np.vstack([l2a_attack_val, verified_feats])

def ae_score(X):
    recon = autoencoder.predict(X, verbose=0)
    return np.mean((X - recon) ** 2, axis=1)

score_n_val = ae_score(l2a_normal_val)
score_attack_val_expanded = ae_score(l2a_attack_val_expanded)
l2a_calib_after = calibrate_threshold(score_n_val, score_attack_val_expanded)
L2A_THRESHOLD_AFTER = l2a_calib_after["threshold"]

print(f"L2A attack-val pool expanded: {len(l2a_attack_val)} -> {len(l2a_attack_val_expanded)}")
print(f"L2A threshold BEFORE: {L2A_THRESHOLD_BEFORE:.6f}")
print(f"L2A threshold AFTER:  {L2A_THRESHOLD_AFTER:.6f}  (FPR={l2a_calib_after['fpr']:.4f}, "
      f"recall={l2a_calib_after['recall']:.4f})")

L2A attack-val pool expanded: 10533 -> 10656
L2A threshold BEFORE: 0.002985
L2A threshold AFTER:  0.002985  (FPR=0.0396, recall=0.8257)


In [11]:
l2a_normal_val = norm.transform(np.load(f"{SPLIT_SRC}/data/splits/l2a_normal_val.npy"))
l2a_attack_val = norm.transform(np.load(f"{SPLIT_SRC}/data/splits/l2a_attack_val.npy"))

verified_feats = np.array([
    norm.transform(to_vector(extract_features({"url": u, "method": "GET", "headers": {}, "body": ""})))[0]
    for u in verified_batch["url"]
])
l2a_attack_val_expanded = np.vstack([l2a_attack_val, verified_feats])

def ae_score(X):
    recon = autoencoder.predict(X, verbose=0)
    return np.mean((X - recon) ** 2, axis=1)

score_n_val = ae_score(l2a_normal_val)
score_attack_val_expanded = ae_score(l2a_attack_val_expanded)
l2a_calib_after = calibrate_threshold(score_n_val, score_attack_val_expanded)
L2A_THRESHOLD_AFTER = l2a_calib_after["threshold"]

print(f"L2A attack-val pool expanded: {len(l2a_attack_val)} -> {len(l2a_attack_val_expanded)}")
print(f"L2A threshold BEFORE: {L2A_THRESHOLD_BEFORE:.6f}")
print(f"L2A threshold AFTER:  {L2A_THRESHOLD_AFTER:.6f}  (FPR={l2a_calib_after['fpr']:.4f}, "
      f"recall={l2a_calib_after['recall']:.4f})")

L2A attack-val pool expanded: 10533 -> 10656
L2A threshold BEFORE: 0.002985
L2A threshold AFTER:  0.002985  (FPR=0.0396, recall=0.8257)


In [12]:
# CHANGED: 4/123 (3.2%) was too small a holdout to trust as a generalization
# test given the larger set. 20% gives a statistically meaningful held-out
# group while still leaving ~100 samples for fine-tuning.
HOLDOUT_FRACTION = 0.20

rng = np.random.RandomState(42)
holdout_idx = rng.choice(verified_batch.index, size=int(len(verified_batch) * HOLDOUT_FRACTION), replace=False)
verified_holdout  = verified_batch.loc[holdout_idx].copy()
verified_finetune = verified_batch.drop(holdout_idx).copy()

print(f"Fine-tuning on {len(verified_finetune)} verified samples")
print(f"Held out {len(verified_holdout)} verified samples for genuine generalization testing")

Fine-tuning on 99 verified samples
Held out 24 verified samples for genuine generalization testing


In [13]:

import torch 
CKPT_PATH = Path(f"{L2B_SRC}/exported_models/layer2b_bigru_checkpoint.pt")
assert CKPT_PATH.exists(), f"{CKPT_PATH} not found"

from layer2b.candidates.bigru import BiGRUClassifier

device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load(CKPT_PATH, map_location=device)
net = BiGRUClassifier(
    vocab_size=ckpt["vocab_size"], embed_dim=ckpt["train_params"]["embed_dim"],
    hidden_dim=ckpt["train_params"]["hidden_dim"], num_layers=ckpt["train_params"]["num_layers"],
    num_classes=ckpt["train_params"]["num_classes"],
).to(device)
net.load_state_dict(ckpt["state_dict"])
print(f"✅ Loaded BiGRU checkpoint (sweep point '{ckpt['sweep_point']}')")

# CHANGED: OVERSAMPLE_FACTOR reduced 30 -> 5. The real sample count already
# grew ~8x (15->124); 30x oversampling on ~100 unique samples would put
# ~3,690 feedback-derived rows into a ~17k-row training set (~18%) — too
# large a fraction, risking the fine-tune skewing toward these near-duplicate
# obfuscation variants at the expense of the rest of the class. 5x keeps the
# feedback contribution proportionate to what worked at the smaller scale.
OVERSAMPLE_FACTOR = 5

X_tok_train_orig = np.load(f"{SPLIT_SRC}/data/splits/l2b_train_X_tokens_sqli8k.npy")
y_train_orig     = np.load(f"{SPLIT_SRC}/data/splits/l2b_train_y_sqli8k.npy")

verified_texts = verified_finetune.apply(
    lambda r: f"{r['method']} {strip_to_path_query(r['url'])} {r['body']}", axis=1
).tolist()
verified_tokens = tok.encode_batch(verified_texts)
verified_tokens_oversampled = np.repeat(verified_tokens, OVERSAMPLE_FACTOR, axis=0)
verified_labels_oversampled = np.repeat(verified_finetune["attack_class_id"].values, OVERSAMPLE_FACTOR)

X_tok_finetune = np.vstack([X_tok_train_orig, verified_tokens_oversampled])
y_finetune     = np.concatenate([y_train_orig, verified_labels_oversampled])

print(f"Fine-tune training set: {len(X_tok_train_orig):,} original + "
      f"{len(verified_tokens_oversampled)} oversampled ({len(verified_finetune)} unique x{OVERSAMPLE_FACTOR}) "
      f"= {len(X_tok_finetune):,} total")

def predict_batched(net, X_tok, device, batch_size=256):
    net.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X_tok), batch_size):
            xb = torch.from_numpy(X_tok[i:i+batch_size]).long().to(device)
            preds.append(net(xb).argmax(1).cpu().numpy())
    return np.concatenate(preds)

X_val_tok = np.load(f"{SPLIT_SRC}/data/splits/l2b_val_X_tokens.npy")
y_val     = np.load(f"{SPLIT_SRC}/data/splits/l2b_val_y.npy")

pre_finetune_val_preds = predict_batched(net, X_val_tok, device)
pre_finetune_val_f1 = f1_score(y_val, pre_finetune_val_preds, average="macro", zero_division=0)
print(f"Original val macro-F1 BEFORE fine-tuning: {pre_finetune_val_f1:.4f}")

holdout_texts = verified_holdout.apply(
    lambda r: f"{r['method']} {strip_to_path_query(r['url'])} {r['body']}", axis=1
).tolist()
X_holdout_tok = tok.encode_batch(holdout_texts)
y_holdout     = verified_holdout["attack_class_id"].values

✅ Loaded BiGRU checkpoint (sweep point '8k')
Fine-tune training set: 17,058 original + 495 oversampled (99 unique x5) = 17,553 total
Original val macro-F1 BEFORE fine-tuning: 0.9937


In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score

CKPT_PATH = Path(f"{L2B_SRC}/exported_models/layer2b_bigru_checkpoint.pt")
assert CKPT_PATH.exists(), f"{CKPT_PATH} not found"

from layer2b.candidates.bigru import BiGRUClassifier
# ... rest of Cell 11 unchanged
FT_EPOCHS = 15
FT_LR = 3e-5
VAL_F1_TOLERANCE = 0.005

tr_dl = DataLoader(
    TensorDataset(torch.from_numpy(X_tok_finetune).long(), torch.from_numpy(y_finetune).long()),
    batch_size=128, shuffle=True,
)

def class_weights_for(y, n_classes, device):
    counts = np.bincount(y, minlength=n_classes).astype(float)
    w = 1.0 / (counts + 1)
    return torch.tensor(w / w.sum() * n_classes, dtype=torch.float32).to(device)

weights = class_weights_for(y_finetune, len(CLASS_NAMES), device)
crit = nn.CrossEntropyLoss(weight=weights)
opt  = torch.optim.AdamW(net.parameters(), lr=FT_LR, weight_decay=1e-5)

epoch_log, epoch_states = [], {}
for epoch in range(1, FT_EPOCHS + 1):
    net.train()
    tr_loss = 0.0
    for xb, yb in tr_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = crit(net(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step()
        tr_loss += loss.item()
    tr_loss /= len(tr_dl)

    val_f1 = f1_score(y_val, predict_batched(net, X_val_tok, device), average="macro", zero_division=0)
    holdout_recall = float((predict_batched(net, X_holdout_tok, device) == y_holdout).mean())

    epoch_log.append({"epoch": epoch, "train_loss": tr_loss, "val_f1": val_f1, "holdout_drift_recall": holdout_recall})
    epoch_states[epoch] = {k: v.clone() for k, v in net.state_dict().items()}
    print(f"  epoch {epoch:2d} | loss={tr_loss:.4f} | val_f1={val_f1:.4f} | holdout_drift_recall={holdout_recall:.4f}")

epoch_df = pd.DataFrame(epoch_log)
eligible = epoch_df[epoch_df["val_f1"] >= (pre_finetune_val_f1 - VAL_F1_TOLERANCE)]

if len(eligible) > 0:
    best_row = eligible.sort_values("holdout_drift_recall", ascending=False).iloc[0]
    status = "ok"
else:
    best_row = epoch_df.sort_values("val_f1", ascending=False).iloc[0]
    status = f"FALLBACK — no epoch within tolerance ({VAL_F1_TOLERANCE}) of baseline"
    print(f"  ⚠️  {status}")

best_epoch = int(best_row["epoch"])
net.load_state_dict(epoch_states[best_epoch])
best_f1 = float(best_row["val_f1"])
best_holdout_recall = float(best_row["holdout_drift_recall"])

print(f"\n✅ Selected epoch {best_epoch}: val_f1={best_f1:.4f} (baseline={pre_finetune_val_f1:.4f}), "
      f"holdout_drift_recall={best_holdout_recall:.4f}")

  epoch  1 | loss=0.2327 | val_f1=0.9919 | holdout_drift_recall=0.7500
  epoch  2 | loss=0.0357 | val_f1=0.9919 | holdout_drift_recall=0.9167
  epoch  3 | loss=0.0160 | val_f1=0.9913 | holdout_drift_recall=0.9167
  epoch  4 | loss=0.0083 | val_f1=0.9931 | holdout_drift_recall=1.0000
  epoch  5 | loss=0.0059 | val_f1=0.9931 | holdout_drift_recall=1.0000
  epoch  6 | loss=0.0052 | val_f1=0.9931 | holdout_drift_recall=1.0000
  epoch  7 | loss=0.0049 | val_f1=0.9931 | holdout_drift_recall=1.0000
  epoch  8 | loss=0.0045 | val_f1=0.9928 | holdout_drift_recall=1.0000
  epoch  9 | loss=0.0041 | val_f1=0.9928 | holdout_drift_recall=1.0000
  epoch 10 | loss=0.0038 | val_f1=0.9928 | holdout_drift_recall=1.0000
  epoch 11 | loss=0.0029 | val_f1=0.9928 | holdout_drift_recall=1.0000
  epoch 12 | loss=0.0034 | val_f1=0.9928 | holdout_drift_recall=1.0000
  epoch 13 | loss=0.0040 | val_f1=0.9925 | holdout_drift_recall=1.0000
  epoch 14 | loss=0.0042 | val_f1=0.9928 | holdout_drift_recall=1.0000
  epoc

In [15]:
holdout_preds_after = predict_batched(net, X_holdout_tok, device)
holdout_recall_after = float((holdout_preds_after == y_holdout).mean())

print("=== AFTER adaptive update — HELD-OUT drift samples (genuine generalization test) ===")
print(f"L2B recall: {holdout_recall_after:.4f}  ({(holdout_preds_after == y_holdout).sum()}/{len(y_holdout)})")

finetune_texts_check = verified_finetune.apply(
    lambda r: f"{r['method']} {strip_to_path_query(r['url'])} {r['body']}", axis=1
).tolist()
finetune_preds = predict_batched(net, tok.encode_batch(finetune_texts_check), device)
memorization_recall = float((finetune_preds == verified_finetune["attack_class_id"].values).mean())
print(f"(Memorization check on fine-tuned rows: {memorization_recall:.4f} — context only, not the claim)")

df_drift["l2a_flagged_after"] = df_drift["l2a_score_before"] >= L2A_THRESHOLD_AFTER
l2a_recall_after = df_drift["l2a_flagged_after"].mean()
print(f"\nL2A recall on full drift set: {l2a_recall_after:.4f}  (was {l2a_recall_before:.4f})")

=== AFTER adaptive update — HELD-OUT drift samples (genuine generalization test) ===
L2B recall: 1.0000  (24/24)
(Memorization check on fine-tuned rows: 0.9899 — context only, not the claim)

L2A recall on full drift set: 1.0000  (was 1.0000)


In [16]:
final_report_d = {
    "drift_scenario": "obfuscated LFI variants (double-encoding, mixed slashes, unicode overlong, null-byte, case/nesting variation) — 124 systematically generated, novel encodings absent from training data",
    "n_drift_samples": len(df_drift),
    "feedback_pipeline": {
        "candidate_pool_size": len(candidate_batch),
        "anti_poison_rejected": int((~candidate_batch["anti_poison_pass"]).sum()),
        "injected_poison_test_caught": bool(poison_caught),
        "human_review_max_batch_ratio": MAX_BATCH_RATIO,
        "human_review_approved": bool(human_review_approved),
        "verified_batch_size": len(verified_batch),
        "finetune_size": len(verified_finetune), "holdout_size": len(verified_holdout),
        "oversample_factor": OVERSAMPLE_FACTOR,
    },
    "l2a": {
        "update_type": "threshold recalibration only — weights unchanged",
        "threshold_before": L2A_THRESHOLD_BEFORE, "threshold_after": L2A_THRESHOLD_AFTER,
        "drift_recall_before": float(l2a_recall_before), "drift_recall_after": float(l2a_recall_after),
    },
    "l2b": {
        "update_type": "fine-tuned from checkpoint, tolerance-banded epoch selection",
        "selected_epoch": best_epoch, "selection_status": status,
        "val_f1_before": float(pre_finetune_val_f1), "val_f1_after": float(best_f1),
        "holdout_drift_recall_before": float(l2b_recall_before),
        "holdout_drift_recall_after": float(holdout_recall_after),
        "memorization_check_recall": float(memorization_recall),
    },
}
with open("results/07_adaptive_retraining_report.json", "w") as f:
    json.dump(final_report_d, f, indent=2)
print(json.dumps(final_report_d, indent=2))

{
  "drift_scenario": "obfuscated LFI variants (double-encoding, mixed slashes, unicode overlong, null-byte, case/nesting variation) \u2014 124 systematically generated, novel encodings absent from training data",
  "n_drift_samples": 124,
  "feedback_pipeline": {
    "candidate_pool_size": 124,
    "anti_poison_rejected": 1,
    "injected_poison_test_caught": true,
    "human_review_max_batch_ratio": 0.1,
    "human_review_approved": true,
    "verified_batch_size": 123,
    "finetune_size": 99,
    "holdout_size": 24,
    "oversample_factor": 5
  },
  "l2a": {
    "update_type": "threshold recalibration only \u2014 weights unchanged",
    "threshold_before": 0.0029852040629296923,
    "threshold_after": 0.002985201298800685,
    "drift_recall_before": 1.0,
    "drift_recall_after": 1.0
  },
  "l2b": {
    "update_type": "fine-tuned from checkpoint, tolerance-banded epoch selection",
    "selected_epoch": 6,
    "selection_status": "ok",
    "val_f1_before": 0.9937273082222333,
    "v